# Silver Layer Transformation

### Purpose

Transform Bronze data into trusted analytical datasets.

**Responsibilities**

- Apply data quality rules
- Standardize datatypes
- Remove unnecessary attributes
- Prepare data for Gold Warehouse

**Input**

bronze_accepted_loans

**Output**

silver_accepted_loans

In [ ]:
df = spark.sql("SELECT * FROM LoanPortfolioLakehouse.dbo.bronze_accepted_loans LIMIT 1000")
display(df)

In [167]:
accepted_df = spark.read.table('bronze_accepted_loans')

rejected_df= spark.read.table('bronze_rejected_loans')

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 169, Finished, Available, Finished, False)

In [168]:
print(f'Accepted Rows: {accepted_df.count():,}')

print(f'Rejected Rows: {rejected_df.count():,}')

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 170, Finished, Available, Finished, False)

Accepted Rows: 2,260,701
Rejected Rows: 27,648,741


In [169]:
from pyspark.sql.functions import col, count, when

def data_quality_summary(df):

    total_rows = df.count()

    results = []

    for column in df.columns:

        null_count = df.filter(
            col(column).isNull() | (col(column) == "")
        ).count()

        distinct_count = df.select(column).distinct().count()

        results.append((
            column,
            null_count,
            round(null_count * 100 / total_rows, 2),
            distinct_count
        ))

    return spark.createDataFrame(
        results,
        [
            "Column",
            "Null Count",
            "Null %",
            "Distinct Values"
        ]
    )

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 171, Finished, Available, Finished, False)

In [170]:
accepted_dq = data_quality_summary(accepted_df)

display(accepted_dq)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 172, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6585a8fc-7727-4477-9319-07589b628507)

In [171]:
silver_accepted_df = accepted_df

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 173, Finished, Available, Finished, False)

In [172]:
# 100 % NULL
silver_accepted_df = silver_accepted_df.drop("member_id")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 174, Finished, Available, Finished, False)

In [173]:
# Validate
print(f"Original Columns : {len(accepted_df.columns)}")
print(f"Silver Columns   : {len(silver_accepted_df.columns)}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 175, Finished, Available, Finished, False)

Original Columns : 151
Silver Columns   : 150


In [174]:
display(
    spark.read.table("silver_accepted_loans").limit(5)
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 176, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56a9b399-a59a-46e6-8b54-f95e3735bf6e)

In [175]:
print(
    spark.read.table("silver_accepted_loans").count()
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 177, Finished, Available, Finished, False)

2260413


In [176]:
# Standardize Percentage columns
from pyspark.sql.functions import regexp_replace, col

percentage_columns = [
    "int_rate",
    "revol_util"
]

for c in percentage_columns:
    silver_accepted_df = silver_accepted_df.withColumn(
        c,
        regexp_replace(col(c), "%", "").cast("double")
    )

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 178, Finished, Available, Finished, False)

In [177]:
# Standardize Loan term
silver_accepted_df = silver_accepted_df.withColumn(
    "term",
    regexp_replace(col("term"), " months", "").cast("int")
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 179, Finished, Available, Finished, False)

In [178]:
# Verify
silver_accepted_df.select(
    "term",
    "int_rate",
    "revol_util"
).show(10, truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 180, Finished, Available, Finished, False)

+----+--------+----------+
|term|int_rate|revol_util|
+----+--------+----------+
|36  |13.99   |29.7      |
|36  |11.99   |19.2      |
|60  |10.78   |56.2      |
|60  |14.85   |11.6      |
|60  |22.45   |64.5      |
|36  |13.44   |68.4      |
|36  |9.17    |84.5      |
|36  |8.49    |5.7       |
|36  |6.49    |34.5      |
|36  |11.48   |39.1      |
+----+--------+----------+
only showing top 10 rows



In [179]:
date_columns = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

silver_accepted_df.select(date_columns).show(10, truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 181, Finished, Available, Finished, False)

+--------+----------------+------------+------------+------------------+
|issue_d |earliest_cr_line|last_pymnt_d|next_pymnt_d|last_credit_pull_d|
+--------+----------------+------------+------------+------------------+
|Dec-2015|Aug-2003        |Jan-2019    |NULL        |Mar-2019          |
|Dec-2015|Dec-1999        |Jun-2016    |NULL        |Mar-2019          |
|Dec-2015|Aug-2000        |Jun-2017    |NULL        |Mar-2019          |
|Dec-2015|Sep-2008        |Feb-2019    |Apr-2019    |Mar-2019          |
|Dec-2015|Jun-1998        |Jul-2016    |NULL        |Mar-2018          |
|Dec-2015|Oct-1987        |May-2017    |NULL        |May-2017          |
|Dec-2015|Jun-1990        |Nov-2016    |NULL        |Mar-2019          |
|Dec-2015|Feb-1999        |Jan-2017    |NULL        |Mar-2019          |
|Dec-2015|Apr-2002        |Aug-2018    |NULL        |Mar-2019          |
|Dec-2015|Nov-1994        |Apr-2017    |NULL        |Nov-2018          |
+--------+----------------+------------+-----------

In [180]:
from pyspark.sql.functions import concat, lit, to_date, col

date_columns = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

for c in date_columns:

    silver_accepted_df = silver_accepted_df.withColumn(
        c,
        to_date(
            concat(
                lit("01-"),
                col(c)
            ),
            "dd-MMM-yyyy"
        )
    )

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 182, Finished, Available, Finished, False)

In [181]:
silver_accepted_df.select(
    date_columns
).show(10, truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 183, Finished, Available, Finished, False)

+----------+----------------+------------+------------+------------------+
|issue_d   |earliest_cr_line|last_pymnt_d|next_pymnt_d|last_credit_pull_d|
+----------+----------------+------------+------------+------------------+
|2015-12-01|2003-08-01      |2019-01-01  |NULL        |2019-03-01        |
|2015-12-01|1999-12-01      |2016-06-01  |NULL        |2019-03-01        |
|2015-12-01|2000-08-01      |2017-06-01  |NULL        |2019-03-01        |
|2015-12-01|2008-09-01      |2019-02-01  |2019-04-01  |2019-03-01        |
|2015-12-01|1998-06-01      |2016-07-01  |NULL        |2018-03-01        |
|2015-12-01|1987-10-01      |2017-05-01  |NULL        |2017-05-01        |
|2015-12-01|1990-06-01      |2016-11-01  |NULL        |2019-03-01        |
|2015-12-01|1999-02-01      |2017-01-01  |NULL        |2019-03-01        |
|2015-12-01|2002-04-01      |2018-08-01  |NULL        |2019-03-01        |
|2015-12-01|1994-11-01      |2017-04-01  |NULL        |2018-11-01        |
+----------+-------------

In [182]:
# ==========================================================
# Data Quality Framework
# ==========================================================

from datetime import datetime

RUN_TIME = datetime.now()

DATASET = "silver_accepted_loans"

from pyspark.sql.functions import col

def dq_check(df, dataset, rule_name, condition):

    failed = df.filter(condition).count()
    total = df.count()
    passed = total - failed
    status = "PASS" if failed == 0 else "FAIL"
    return (

        RUN_TIME,
        dataset,
        rule_name,
        total,
        passed,
        failed,
        status

    )

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 184, Finished, Available, Finished, False)

In [183]:
# First 5 rules
dq_results = []

dq_results.append(
    dq_check(
        silver_accepted_df,
        DATASET,
        "Loan ID Missing",
        col("id").isNull()
    )
)

dq_results.append(
    dq_check(
        silver_accepted_df,
        DATASET,
        "Loan Amount <= 0",
        col("loan_amnt") <= 0
    )
)

dq_results.append(
    dq_check(
        silver_accepted_df,
        DATASET,
        "Interest Rate Missing",
        col("int_rate").isNull()
    )
)

dq_results.append(
    dq_check(
        silver_accepted_df,
        DATASET,
        "Grade Missing",
        col("grade").isNull()
    )
)

dq_results.append(
    dq_check(
        silver_accepted_df,
        DATASET,
        "Issue Date Missing",
        col("issue_d").isNull()
    )
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 185, Finished, Available, Finished, False)

In [184]:
# Convert into a dataframe

dq_df = spark.createDataFrame(
    dq_results,
    [
        "Run Timestamp",
        "Dataset",
        "Rule",
        "Rows Checked",
        "Passed",
        "Failed",
        "Status"
    ]
)

display(dq_df)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 186, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2832124-7450-4cad-b5e2-624328ad67e8)

In [185]:
from pyspark.sql import functions as F

total_rows = silver_accepted_df.count()

distinct_ids = (
    silver_accepted_df
    .select("id")
    .distinct()
    .count()
)

print(f"Total Rows   : {total_rows:,}")
print(f"Distinct IDs : {distinct_ids:,}")

if total_rows == distinct_ids:
    print("✅ Loan IDs are unique.")
else:
    print(f"❌ Duplicate Loan IDs : {total_rows - distinct_ids:,}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 187, Finished, Available, Finished, False)

Total Rows   : 2,260,701
Distinct IDs : 2,260,701
✅ Loan IDs are unique.


In [186]:
duplicate_rows = (
    silver_accepted_df.count()
    -
    silver_accepted_df.distinct().count()
)
print(f"Duplicate Rows : {duplicate_rows}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 188, Finished, Available, Finished, False)

Duplicate Rows : 0


### Business Rule 1
Loan amount should always be greater than zero.

In [187]:
invalid_loan_amount = (
    silver_accepted_df
    .filter(F.col("loan_amnt") <= 0)
)

print(f"Invalid Loan Amount Records : {invalid_loan_amount.count():,}")

display(
    invalid_loan_amount.select(
        "id",
        "loan_amnt"
    )
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 189, Finished, Available, Finished, False)

Invalid Loan Amount Records : 0


SynapseWidget(Synapse.DataFrame, 5b3f45ef-2ece-463a-be95-f7ae768bc87c)

###  Rule 2 — Funded Amount cannot exceed Requested Amount

**Business meaning:**

A borrower cannot receive more money than they requested.

In [188]:
invalid_funding = (
    silver_accepted_df
    .filter(
        F.col("funded_amnt") > F.col("loan_amnt")
    )
)

print(f"Invalid Funding Records : {invalid_funding.count():,}")

display(
    invalid_funding.select(
        "id",
        "loan_amnt",
        "funded_amnt"
    )
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 190, Finished, Available, Finished, False)

Invalid Funding Records : 0


SynapseWidget(Synapse.DataFrame, 8025e45b-fe8d-4da8-b056-98e8672206b2)

### Rule 3 — Interest Rate

Interest rates should be within a reasonable lending range.

In [189]:
invalid_interest = (
    silver_accepted_df
    .filter(
        (F.col("int_rate") <= 0) |
        (F.col("int_rate") > 40)
    )
)

print(f"Invalid Interest Rate Records : {invalid_interest.count():,}")

display(
    invalid_interest.select(
        "id",
        "int_rate"
    )
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 191, Finished, Available, Finished, False)

Invalid Interest Rate Records : 0


SynapseWidget(Synapse.DataFrame, 0d4efbc3-1c13-4e49-a90d-f6bf0a15d593)

### Rule 4 — Loan Term

In [190]:
invalid_term = (
    silver_accepted_df
    .filter(
        ~F.col("term").isin(36, 60)
    )
)

print(f"Invalid Terms : {invalid_term.count():,}")

display(
    invalid_term.select(
        "id",
        "term"
    )
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 192, Finished, Available, Finished, False)

Invalid Terms : 0


SynapseWidget(Synapse.DataFrame, e1d0f3c2-3ede-4902-8477-0cb768464a53)

### Rule 5 — Grade Validation

In [191]:
silver_accepted_df.groupBy("grade") \
    .count() \
    .orderBy("grade") \
    .show()

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 193, Finished, Available, Finished, False)

+-----+------+
|grade| count|
+-----+------+
| NULL|    33|
|    A|433027|
|    B|663557|
|    C|650053|
|    D|324424|
|    E|135639|
|    F| 41800|
|    G| 12168|
+-----+------+



### Rule 6 — Sub Grade Validation

In [192]:
silver_accepted_df.groupBy("sub_grade") \
    .count() \
    .orderBy("sub_grade") \
    .show(40, truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 194, Finished, Available, Finished, False)

+---------+------+
|sub_grade|count |
+---------+------+
|NULL     |33    |
|A1       |86790 |
|A2       |69562 |
|A3       |73184 |
|A4       |95874 |
|A5       |107617|
|B1       |125341|
|B2       |126621|
|B3       |131514|
|B4       |139793|
|B5       |140288|
|C1       |145903|
|C2       |131116|
|C3       |129193|
|C4       |127115|
|C5       |116726|
|D1       |81787 |
|D2       |72899 |
|D3       |64819 |
|D4       |56896 |
|D5       |48023 |
|E1       |33573 |
|E2       |29924 |
|E3       |26708 |
|E4       |22763 |
|E5       |22671 |
|F1       |13413 |
|F2       |9305  |
|F3       |7791  |
|F4       |6124  |
|F5       |5167  |
|G1       |4106  |
|G2       |2688  |
|G3       |2094  |
|G4       |1712  |
|G5       |1568  |
+---------+------+



### Rule 7 — Loan Status Validation

In [193]:
silver_accepted_df.groupBy("loan_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 195, Finished, Available, Finished, False)

+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1076751|
|Current                                            |878317 |
|Charged Off                                        |268558 |
|Late (31-120 days)                                 |21467  |
|In Grace Period                                    |8436   |
|Late (16-30 days)                                  |4349   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |40     |
|NULL                                               |33     |
|Oct-2015                                           |1      |
+---------------------------------------------------+-------+



### Rule 8 — Home Ownership

In [194]:
silver_accepted_df.groupBy("home_ownership") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 196, Finished, Available, Finished, False)

+--------------+-------+
|home_ownership|count  |
+--------------+-------+
|MORTGAGE      |1111449|
|RENT          |894929 |
|OWN           |253057 |
|ANY           |996    |
|OTHER         |182    |
|NONE          |54     |
|NULL          |33     |
|2 years       |1      |
+--------------+-------+



### Rule 9 — Verification Status

In [195]:
silver_accepted_df.groupBy("verification_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 197, Finished, Available, Finished, False)

+-------------------+------+
|verification_status|count |
+-------------------+------+
|Source Verified    |886230|
|Not Verified       |744806|
|Verified           |629631|
|NULL               |33    |
|38000.0            |1     |
+-------------------+------+



In [196]:
display(

    silver_accepted_df.filter(

        F.col("loan_status") == "Oct-2015"

    )

)
display(

    silver_accepted_df.filter(

        F.col("home_ownership") == "2 years"

    )

)
display(

    silver_accepted_df.filter(

        F.col("verification_status") == "38000.0"

    )

)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 198, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d04ed274-55e2-4128-8e66-97e9627a6a06)

SynapseWidget(Synapse.DataFrame, 58e99287-e0c3-49ee-b877-2d46bc48f2d0)

SynapseWidget(Synapse.DataFrame, ee100b34-f884-4d17-92ff-c40b5facc631)

**During Silver validation I detected a malformed source record because three unrelated categorical columns each contained one unexpected value.**

Investigation showed they belonged to the same row, indicating a source parsing issue.

## Removing malformed source record

During categorical validation, one loan (ID = 61400928) contained
misaligned values across multiple business columns:

• loan_status = Oct-2015
• home_ownership = 2 years
• verification_status = 38000.0

**The row appears to be malformed during source ingestion.**

Since the correct values cannot be reconstructed confidently,
the record is **removed from the Silver layer.**

In [197]:
silver_accepted_df = silver_accepted_df.filter(
    F.col("id") != 61400928
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 199, Finished, Available, Finished, False)

In [198]:
print(silver_accepted_df.count())

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 200, Finished, Available, Finished, False)

2260667


### Current Silver Progress
✅ **Completed**   
Bronze loaded   
Bronze profiled 
Data dictionary started  
Data types corrected        
Dates converted  
Percentage columns cleaned  
member_id removed (100% null)   
Duplicate checks    
Business rule validations   
Category validations    
Malformed record removed    

In [199]:
silver_accepted_df = spark.read.table("silver_accepted_loans")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 201, Finished, Available, Finished, False)

In [200]:
from pyspark.sql import functions as F

total_rows = silver_accepted_df.count()
null_profile = silver_accepted_df.select(
    *[
        (
            (
                F.count(
                    F.when(
                        F.col(c).isNull() | (F.col(c) == ""),
                        c
                    )
                ) / total_rows * 100
            ).alias(c)
        )
        for c in silver_accepted_df.columns
    ]
)

null_profile = (
    null_profile
    .selectExpr(
        "stack({}, {}) as (Column, `Null %`)".format(
            len(silver_accepted_df.columns),
            ",".join(
                [f"'{c}', `{c}`" for c in silver_accepted_df.columns]
            )
        )
    )
    .orderBy(F.desc("Null %"))
)

display(null_profile)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 202, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c4d467b6-8008-4383-81ac-f17749292e90)

In [201]:
for c in silver_accepted_df.columns:
    print(c)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 203, Finished, Available, Finished, False)

id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
purpose
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
application_type
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_within_12_mths
delinq_amnt
mo_sin_old_il_acct
mo_sin_ol

# Removing Non-Analytical Columns

These columns are removed because they either:

- contain free text,
- expose PII,
- are hyperlinks,
- or provide no analytical value.

Their removal simplifies downstream analytics while preserving business measures.

In [202]:
drop_columns = [

    "url",
    "desc",
    "member_id",
    "emp_title"

]

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 204, Finished, Available, Finished, False)

In [203]:
silver_accepted_df = silver_accepted_df.drop(*drop_columns)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 205, Finished, Available, Finished, False)

In [204]:
print(len(silver_accepted_df.columns))

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 206, Finished, Available, Finished, False)

104


In [205]:
drop_columns = [

    # Free text
    "title",

    # LendingClub webpage metadata
    "pymnt_plan",

    # Legacy policy
    "policy_code"

]

silver_accepted_df = silver_accepted_df.drop(*drop_columns)

print(len(silver_accepted_df.columns))

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 207, Finished, Available, Finished, False)

104


# Removing Servicing-Specific Columns

The objective of this project is portfolio analytics rather than loan servicing.

The following feature groups relate to hardship programs,
settlement workflows, payment deferrals, and secondary applicants.

These attributes are retained in the Bronze layer but excluded from
the Silver analytical model because they do not contribute to the
planned business dashboards or KPIs.

In [206]:
hardship_columns = [

    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount"

]

silver_accepted_df = silver_accepted_df.drop(*hardship_columns)

print(f"Columns remaining : {len(silver_accepted_df.columns)}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 208, Finished, Available, Finished, False)

Columns remaining : 104


## Removing Settlement Workflow Columns

Settlement-related columns describe the operational debt settlement
process rather than loan origination or portfolio performance.

Since the goal of this project is portfolio analytics,
these fields are excluded from the Silver analytical model.

In [207]:
settlement_columns = [

    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"

]

silver_accepted_df = silver_accepted_df.drop(*settlement_columns)

print(f"Columns remaining : {len(silver_accepted_df.columns)}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 209, Finished, Available, Finished, False)

Columns remaining : 104


In [208]:
joint_application_columns = [

    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",

    "sec_app_fico_range_low",
    "sec_app_fico_range_high",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"

]

silver_accepted_df = silver_accepted_df.drop(*joint_application_columns)

print(f"Columns remaining : {len(silver_accepted_df.columns)}")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 210, Finished, Available, Finished, False)

Columns remaining : 104


In [209]:
final_drop_columns = [

    "zip_code",      # State is enough
    "title",         # Free text
    "collection_recovery_fee",
    "policy_code"

]

silver_accepted_df = silver_accepted_df.drop(*final_drop_columns)

print(len(silver_accepted_df.columns))

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 211, Finished, Available, Finished, False)

104


In [210]:
print("=" * 50)
print("SILVER LAYER SUMMARY")
print("=" * 50)

print(f"Rows    : {silver_accepted_df.count():,}")
print(f"Columns : {len(silver_accepted_df.columns)}")

print("\nSchema\n")

silver_accepted_df.printSchema()

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 212, Finished, Available, Finished, False)

SILVER LAYER SUMMARY
Rows    : 2,260,413
Columns : 104

Schema

root
 |-- id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: integer (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: date (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- delinq_2yrs: string (nullable = true)
 |-- earliest_cr_line: date (nullable = true)
 |-- fico_range_low: string (nullable = true)
 |-- fico_range_high: string (nullable = true)
 |-- inq_

In [211]:
print(silver_accepted_df.count())
print(len(silver_accepted_df.columns))

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 213, Finished, Available, Finished, False)

2260413
104


In [212]:
spark.read.table("silver_accepted_loans") \
    .filter(F.col("id") == 61400928) \
    .show()

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 214, Finished, Available, Finished, False)

+---+---------+-----------+---------------+----+--------+-----------+-----+---------+----------+--------------+----------+-------------------+-------+-----------+-------+----------+---+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+-----------+---------------+---------------+-------------+------------------+----------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+----------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+------------+--------------------+-----------+--------------+-------+------------------------+-----------+------------------+-------

In [213]:
spark.read.table("silver_accepted_loans") \
    .filter(F.col("id") == 61400928) \
    .count()

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 215, Finished, Available, Finished, False)

0

In [214]:
spark.read.table("silver_accepted_loans") \
    .filter(F.col("id") == 61400928) \
    .select(
        "id",
        "issue_d",
        "loan_status",
        "home_ownership",
        "verification_status"
    ) \
    .show(truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 216, Finished, Available, Finished, False)

+---+-------+-----------+--------------+-------------------+
|id |issue_d|loan_status|home_ownership|verification_status|
+---+-------+-----------+--------------+-------------------+
+---+-------+-----------+--------------+-------------------+



In [215]:
verify = spark.read.table("silver_accepted_loans")

verify.filter(
    F.col("id") == 61400928
).count()

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 217, Finished, Available, Finished, False)

0

In [216]:
valid_purposes = [
    "debt_consolidation",
    "credit_card",
    "home_improvement",
    "other",
    "major_purchase",
    "medical",
    "small_business",
    "car",
    "vacation",
    "moving",
    "house",
    "wedding",
    "renewable_energy",
    "educational"
]

silver_accepted_df = silver_accepted_df.filter(
    F.col("purpose").isin(valid_purposes)
)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 218, Finished, Available, Finished, False)

In [217]:
silver_accepted_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_accepted_loans")

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 219, Finished, Available, Finished, False)

In [218]:
spark.read.table("silver_accepted_loans") \
    .groupBy("purpose") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

StatementMeta(, 7ebf60db-2629-4c9c-ad2c-9b6b5b0ec2be, 220, Finished, Available, Finished, False)

+------------------+-------+
|purpose           |count  |
+------------------+-------+
|debt_consolidation|1277790|
|credit_card       |516926 |
|home_improvement  |150440 |
|other             |139413 |
|major_purchase    |50429  |
|medical           |27481  |
|small_business    |24659  |
|car               |24009  |
|vacation          |15525  |
|moving            |15402  |
|house             |14131  |
|wedding           |2351   |
|renewable_energy  |1445   |
|educational       |412    |
+------------------+-------+



In [1]:
from pyspark.sql.functions import current_timestamp

log_df = spark.createDataFrame([
    (
        "pl_loan_portfolio_etl",
        "nb_transform_silver",
        "SUCCESS",
        silver_accepted_df.count()
    )
], ["pipeline_name", "notebook_name", "status", "rows_processed"])

log_df = log_df.withColumn("run_time", current_timestamp())

log_df.write.mode("append").saveAsTable("audit_log")

StatementMeta(, b096f19f-7c07-47c8-8c86-6cf0b1768d85, 3, Finished, Available, Finished, False)

NameError: name 'silver_accepted_df' is not defined